# TSLA Stock Price Prediction
End-to-end pipeline: data download, feature engineering, model training (Linear Regression, Decision Tree, Random Forest), evaluation, visualization, and model saving.

## 1. Imports

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import yfinance as yf
import joblib
import warnings

from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

warnings.filterwarnings("ignore")
sns.set(style="darkgrid")

## 2. Data Download

In [ ]:
def download_data(ticker="TSLA", period="5y"):
    """Download historical stock data using yfinance."""
    df = yf.download(ticker, period=period)
    df.reset_index(inplace=True)
    return df

## 3. Feature Engineering Functions

In [ ]:
def add_lag_features(df, lags=[1, 2, 3, 5, 7, 14]):
    """Add lagged closing price features."""
    for lag in lags:
        df[f"Lag_{lag}"] = df["Close"].shift(lag)
    return df

In [ ]:
def add_moving_averages(df):
    """Add SMA and EMA features."""
    df["SMA_5"] = df["Close"].rolling(window=5).mean()
    df["SMA_10"] = df["Close"].rolling(window=10).mean()
    df["SMA_20"] = df["Close"].rolling(window=20).mean()

    df["EMA_5"] = df["Close"].ewm(span=5, adjust=False).mean()
    df["EMA_10"] = df["Close"].ewm(span=10, adjust=False).mean()
    df["EMA_20"] = df["Close"].ewm(span=20, adjust=False).mean()
    return df

In [ ]:
def add_rsi(df, period=14):
    """Add Relative Strength Index (RSI)."""
    delta = df["Close"].diff()
    gain = delta.where(delta > 0, 0)
    loss = -delta.where(delta < 0, 0)

    avg_gain = gain.rolling(window=period).mean()
    avg_loss = loss.rolling(window=period).mean()

    rs = avg_gain / avg_loss
    df["RSI"] = 100 - (100 / (1 + rs))
    return df

In [ ]:
def add_macd(df):
    """Add MACD and Signal line."""
    ema_12 = df["Close"].ewm(span=12, adjust=False).mean()
    ema_26 = df["Close"].ewm(span=26, adjust=False).mean()
    df["MACD"] = ema_12 - ema_26
    df["MACD_Signal"] = df["MACD"].ewm(span=9, adjust=False).mean()
    return df

In [ ]:
def add_bollinger_bands(df, window=20):
    """Add Bollinger Bands (Upper and Lower)."""
    sma = df["Close"].rolling(window=window).mean()
    std = df["Close"].rolling(window=window).std()
    df["BB_Upper"] = sma + (2 * std)
    df["BB_Lower"] = sma - (2 * std)
    return df

In [ ]:
def add_momentum(df, period=10):
    """Add Momentum indicator."""
    df["Momentum"] = df["Close"] - df["Close"].shift(period)
    return df

In [ ]:
def add_returns_and_volatility(df, window=10):
    """Add Daily Return and Volatility."""
    df["Daily_Return"] = df["Close"].pct_change()
    df["Volatility"] = df["Daily_Return"].rolling(window=window).std()
    return df

In [ ]:
def create_target(df):
    """Create target variable: next-day closing price."""
    df["Target"] = df["Close"].shift(-1)
    return df

In [ ]:
def engineer_features(df):
    """Run all feature engineering steps."""
    df = add_lag_features(df)
    df = add_moving_averages(df)
    df = add_rsi(df)
    df = add_macd(df)
    df = add_bollinger_bands(df)
    df = add_momentum(df)
    df = add_returns_and_volatility(df)
    df = create_target(df)
    return df

## 4. Preprocessing

In [ ]:
def preprocess_data(df):
    """Clean and prepare the dataframe."""
    # Flatten multi-index columns if present (yfinance sometimes returns multiindex)
    if isinstance(df.columns, pd.MultiIndex):
        df.columns = [col[0] if col[1] == "" else col[0] for col in df.columns]

    # Convert Date column to datetime
    df["Date"] = pd.to_datetime(df["Date"])

    # Sort by date
    df.sort_values("Date", inplace=True)

    # Remove duplicates
    df.drop_duplicates(inplace=True)

    # Reset index after sorting
    df.reset_index(drop=True, inplace=True)

    # Feature engineering
    df = engineer_features(df)

    # Handle missing values created by rolling/lag features
    df.dropna(inplace=True)
    df.reset_index(drop=True, inplace=True)

    return df

## 5. Train-Test Split

In [ ]:
def time_based_split(df, feature_cols, target_col="Target", split_ratio=0.8):
    """Perform a time-based train-test split (no shuffling)."""
    split_index = int(len(df) * split_ratio)

    X_train = df.iloc[:split_index][feature_cols]
    X_test = df.iloc[split_index:][feature_cols]

    y_train = df.iloc[:split_index][target_col]
    y_test = df.iloc[split_index:][target_col]

    dates_test = df.iloc[split_index:]["Date"]

    return X_train, X_test, y_train, y_test, dates_test

## 6. Linear Regression

In [ ]:
def train_linear_regression(X_train, y_train):
    """Train a simple Linear Regression model."""
    model = LinearRegression()
    model.fit(X_train, y_train)
    return model

## 7. Decision Tree (with GridSearchCV)

In [ ]:
def train_decision_tree(X_train, y_train):
    """Train Decision Tree Regressor with hyperparameter tuning."""
    param_grid = {
        "max_depth": [3, 5, 7, 10, None],
        "min_samples_split": [2, 5, 10],
        "min_samples_leaf": [1, 2, 4]
    }

    dt = DecisionTreeRegressor(random_state=42)
    grid_search = GridSearchCV(
        estimator=dt,
        param_grid=param_grid,
        cv=5,
        scoring="neg_mean_squared_error",
        n_jobs=-1
    )
    grid_search.fit(X_train, y_train)
    print("Best Decision Tree Params:", grid_search.best_params_)
    return grid_search.best_estimator_

## 8. Random Forest (with GridSearchCV)

In [ ]:
def train_random_forest(X_train, y_train):
    """Train Random Forest Regressor with hyperparameter tuning."""
    param_grid = {
        "n_estimators": [50, 100, 200],
        "max_depth": [5, 10, None],
        "min_samples_split": [2, 5],
        "min_samples_leaf": [1, 2]
    }

    rf = RandomForestRegressor(random_state=42)
    grid_search = GridSearchCV(
        estimator=rf,
        param_grid=param_grid,
        cv=5,
        scoring="neg_mean_squared_error",
        n_jobs=-1
    )
    grid_search.fit(X_train, y_train)
    print("Best Random Forest Params:", grid_search.best_params_)
    return grid_search.best_estimator_

## 9. Evaluation Functions

In [ ]:
def evaluate_model(model, X_test, y_test, model_name):
    """Evaluate a model using MAE, MSE, RMSE, and R2 Score."""
    y_pred = model.predict(X_test)

    mae = mean_absolute_error(y_test, y_pred)
    mse = mean_squared_error(y_test, y_pred)
    rmse = np.sqrt(mse)
    r2 = r2_score(y_test, y_pred)

    metrics = {
        "Model": model_name,
        "MAE": mae,
        "MSE": mse,
        "RMSE": rmse,
        "R2_Score": r2
    }

    return metrics, y_pred

In [ ]:
def build_metrics_table(metrics_list):
    """Build a comparison table for all models."""
    return pd.DataFrame(metrics_list)

In [ ]:
def select_best_model(metrics_df, models_dict):
    """Select the best model based on lowest RMSE and highest R2."""
    best_row = metrics_df.sort_values(by=["RMSE", "R2_Score"], ascending=[True, False]).iloc[0]
    best_model_name = best_row["Model"]
    best_model = models_dict[best_model_name]
    print(f"\nBest Model Selected: {best_model_name}")
    print(best_row)
    return best_model_name, best_model

## 10. Visualizations

In [ ]:
def plot_close_price_trend(df):
    """Plot TSLA closing price trend over time."""
    plt.figure(figsize=(14, 6))
    plt.plot(df["Date"], df["Close"], color="blue", label="Close Price")
    plt.title("TSLA Close Price Trend")
    plt.xlabel("Date")
    plt.ylabel("Price (USD)")
    plt.legend()
    plt.tight_layout()
    plt.savefig("tsla_close_price_trend.png")
    plt.show()

In [ ]:
def plot_correlation_heatmap(df, feature_cols, target_col="Target"):
    """Plot correlation heatmap of features and target."""
    plt.figure(figsize=(16, 12))
    corr_cols = feature_cols + [target_col]
    corr = df[corr_cols].corr()
    sns.heatmap(corr, annot=False, cmap="coolwarm", center=0)
    plt.title("Correlation Heatmap")
    plt.tight_layout()
    plt.savefig("correlation_heatmap.png")
    plt.show()

In [ ]:
def plot_actual_vs_predicted(dates_test, y_test, predictions_dict):
    """Plot actual vs predicted values for all models."""
    plt.figure(figsize=(14, 6))
    plt.plot(dates_test, y_test.values, label="Actual", color="black", linewidth=2)

    for model_name, y_pred in predictions_dict.items():
        plt.plot(dates_test, y_pred, label=f"Predicted ({model_name})", linestyle="--")

    plt.title("Actual vs Predicted Close Price")
    plt.xlabel("Date")
    plt.ylabel("Price (USD)")
    plt.legend()
    plt.tight_layout()
    plt.savefig("actual_vs_predicted.png")
    plt.show()

In [ ]:
def plot_residuals(y_test, y_pred, model_name):
    """Plot residuals for a given model."""
    residuals = y_test.values - y_pred

    plt.figure(figsize=(12, 5))
    plt.scatter(range(len(residuals)), residuals, alpha=0.6, color="purple")
    plt.axhline(y=0, color="red", linestyle="--")
    plt.title(f"Residual Plot - {model_name}")
    plt.xlabel("Test Sample Index")
    plt.ylabel("Residual (Actual - Predicted)")
    plt.tight_layout()
    plt.savefig(f"residual_plot_{model_name.replace(' ', '_')}.png")
    plt.show()

In [ ]:
def plot_feature_importance(model, feature_cols, model_name):
    """Plot feature importance for tree-based models."""
    if not hasattr(model, "feature_importances_"):
        print(f"{model_name} does not support feature importance.")
        return

    importances = model.feature_importances_
    feat_imp_df = pd.DataFrame({
        "Feature": feature_cols,
        "Importance": importances
    }).sort_values(by="Importance", ascending=False)

    plt.figure(figsize=(12, 8))
    sns.barplot(x="Importance", y="Feature", data=feat_imp_df, palette="viridis")
    plt.title(f"Feature Importance - {model_name}")
    plt.tight_layout()
    plt.savefig(f"feature_importance_{model_name.replace(' ', '_')}.png")
    plt.show()

In [ ]:
def plot_model_comparison(metrics_df):
    """Plot bar chart comparing model performance (RMSE and R2)."""
    fig, axes = plt.subplots(1, 2, figsize=(14, 6))

    sns.barplot(x="Model", y="RMSE", data=metrics_df, ax=axes[0], palette="Blues_d")
    axes[0].set_title("RMSE Comparison")
    axes[0].tick_params(axis="x", rotation=20)

    sns.barplot(x="Model", y="R2_Score", data=metrics_df, ax=axes[1], palette="Greens_d")
    axes[1].set_title("R2 Score Comparison")
    axes[1].tick_params(axis="x", rotation=20)

    plt.tight_layout()
    plt.savefig("model_comparison.png")
    plt.show()

## 11. Model Saving

In [ ]:
def save_best_model(best_model, best_model_name, filename="best_model.pkl"):
    """Save the best model using joblib."""
    joblib.dump(best_model, filename)
    print(f"Best model ({best_model_name}) saved as {filename}")

In [ ]:
def save_predictions(dates_test, y_test, predictions_dict, filename="predictions.csv"):
    """Save prediction results to CSV."""
    results_df = pd.DataFrame({
        "Date": dates_test.values,
        "Actual": y_test.values
    })

    for model_name, y_pred in predictions_dict.items():
        results_df[f"Predicted_{model_name.replace(' ', '_')}"] = y_pred

    results_df.to_csv(filename, index=False)
    print(f"Predictions saved to {filename}")

## 12. Run the Pipeline
Instead of one big `main()` function, each step below is its own cell so you can run and inspect the pipeline step by step.

### Step 1: Download data

In [ ]:
df = download_data(ticker="TSLA", period="5y")
df.head()

### Step 2: Preprocess and engineer features

In [ ]:
df = preprocess_data(df)
df.head()

### Step 3: Define feature columns

In [ ]:
exclude_cols = ["Date", "Target", "Close", "Adj Close"]
feature_cols = [col for col in df.columns if col not in exclude_cols]
feature_cols

### Step 4: Train-test split (time-based)

In [ ]:
X_train, X_test, y_train, y_test, dates_test = time_based_split(
    df, feature_cols, target_col="Target", split_ratio=0.8
)
print(X_train.shape, X_test.shape)

### Step 5: Train models

In [ ]:
print("Training Linear Regression...")
lr_model = train_linear_regression(X_train, y_train)

print("Training Decision Tree with GridSearchCV...")
dt_model = train_decision_tree(X_train, y_train)

print("Training Random Forest with GridSearchCV...")
rf_model = train_random_forest(X_train, y_train)

models_dict = {
    "Linear Regression": lr_model,
    "Decision Tree": dt_model,
    "Random Forest": rf_model
}

### Step 6: Evaluate models

In [ ]:
metrics_list = []
predictions_dict = {}

for model_name, model in models_dict.items():
    metrics, y_pred = evaluate_model(model, X_test, y_test, model_name)
    metrics_list.append(metrics)
    predictions_dict[model_name] = y_pred

metrics_df = build_metrics_table(metrics_list)
print("\nModel Performance Comparison:")
metrics_df

### Step 7: Select best model

In [ ]:
best_model_name, best_model = select_best_model(metrics_df, models_dict)

### Step 8: Visualizations

In [ ]:
plot_close_price_trend(df)

In [ ]:
plot_correlation_heatmap(df, feature_cols, target_col="Target")

In [ ]:
plot_actual_vs_predicted(dates_test, y_test, predictions_dict)

In [ ]:
for model_name in models_dict.keys():
    plot_residuals(y_test, predictions_dict[model_name], model_name)
    plot_feature_importance(models_dict[model_name], feature_cols, model_name)

In [ ]:
plot_model_comparison(metrics_df)

### Step 9: Save best model and predictions

In [ ]:
save_best_model(best_model, best_model_name, filename="best_model.pkl")
save_predictions(dates_test, y_test, predictions_dict, filename="predictions.csv")

### Step 10: Save metrics table

In [ ]:
metrics_df.to_csv("model_metrics_comparison.csv", index=False)
print("\nMetrics table saved to model_metrics_comparison.csv")